In [80]:
import os

In [81]:
%pwd

'c:\\ALI\\Kidney-Disease-Classification-Deep-Learning-Project'

In [82]:
from pathlib import Path

workspace_name = "Kidney-Disease-Classification-Deep-Learning-Project"
workspace_candidates = [
    Path.cwd(),
    Path.cwd() / "ALI" / workspace_name,
    Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd(),
]
project_root = next(
    (candidate for candidate in workspace_candidates if (candidate / "config" / "config.yaml").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing config/config.yaml")
os.chdir(project_root)

In [83]:
%pwd

'c:\\ALI\\Kidney-Disease-Classification-Deep-Learning-Project'

In [84]:
from dataclasses import dataclass
from pathlib import Path

In [91]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [86]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [87]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [88]:
import os
import zipfile
import gdown
from cnnClassifier import logger

In [89]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [90]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
        raise e

[2026-09-13 09:32:47,346: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-13 09:32:47,349: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-13 09:32:47,350: INFO: common: created directory at: artifacts]
[2026-09-13 09:32:47,353: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-13 09:32:47,355: INFO: 986331348: Downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=45c3861a-1680-40df-b51f-58ed55fc0cf4
To: c:\ALI\Kidney-Disease-Classification-Deep-Learning-Project\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:05<00:00, 11.0MB/s]

[2026-09-13 09:32:56,058: INFO: 986331348: Downloaded data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]
